# 🎮 Sarcasm Detection — Method 4: Live Demo (All 3 Methods)
**Project:** `Sarcasm-Detection-NLP` | **Course:** CSE3246 — Natural Language Processing  
**Dataset:** `raquiba/Sarcasm_News_Headline`  
**Notebook:** `04_demo.ipynb` — Interactive ipywidgets demo: type any headline → compare all 3 models

---
### Prerequisites
Run this notebook **after** all three method notebooks:
- `01_rule_based.ipynb` → produces `outputs/method1/method1_rulebased_results.json`
- `02_tfidf_sklearn.ipynb` → produces `outputs/method2_tfidf_results.json`
- `03_bert_finetune.ipynb` → produces `outputs/method3/` checkpoint

### Architecture
```
Section 1 │ Install & Imports
Section 2 │ Load & retrain TF-IDF + SVM  (fast, <2 min)
Section 3 │ Load BERT checkpoint
Section 4 │ Load Rule Engine  (from 01_rule_based.ipynb)
Section 5 │ Predict functions
Section 6 │ ipywidgets interactive demo
Section 7 │ Batch evaluation on curated examples
```

## Section 1 — Install & Imports

In [ ]:
# ── Install dependencies (Colab) ──────────────────────────────────────────────
!pip install spacy datasets transformers torch scikit-learn pandas numpy ipywidgets --quiet
!python -m spacy download en_core_web_sm -q
# Enable ipywidgets in Colab
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import re, json, warnings, time
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path

import spacy
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizerFast, BertForSequenceClassification

warnings.filterwarnings('ignore')

# ── Constants — match all three notebooks ────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Paths — must match where 01/02/03 saved their outputs ────────────────────
OUTPUT_DIR = Path('outputs')
M3_CKPT    = OUTPUT_DIR / 'method3' / 'checkpoint'   # saved by 03_bert_finetune.ipynb
M2_JSON    = OUTPUT_DIR / 'method2_tfidf_results.json' # saved by 02_tfidf_sklearn.ipynb
M1_JSON    = OUTPUT_DIR / 'method1' / 'method1_rulebased_results.json'  # saved by 01_rule_based.ipynb

print(f'✅ Imports OK | Device: {DEVICE}')
print(f'   BERT checkpoint expected at : {M3_CKPT}')
print(f'   TF-IDF results JSON at      : {M2_JSON}')

## Section 2 — Load Dataset & Retrain TF-IDF + SVM

> We retrain the SVM in-session rather than pickling, so the demo works in
> a fresh Colab runtime. Uses **identical hyperparams** to `02_tfidf_sklearn.ipynb`.

In [ ]:
# ── Load dataset — same pipeline as 02_tfidf_sklearn.ipynb ───────────────────
print('⏳ Loading raquiba/Sarcasm_News_Headline...')
raw = load_dataset('raquiba/Sarcasm_News_Headline')

dfs = []
for split_name, split_data in raw.items():
    tmp = split_data.to_pandas(); tmp['split'] = split_name; dfs.append(tmp)
df_raw = pd.concat(dfs, ignore_index=True)

label_col = 'is_sarcastic' if 'is_sarcastic' in df_raw.columns else df_raw.columns[-1]
text_col  = 'headline'     if 'headline'     in df_raw.columns else df_raw.columns[0]

df = df_raw[[text_col, label_col]].copy()
df.columns = ['headline', 'label']
df = df.dropna(subset=['headline','label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

# ── Preprocessing — same clean_headline as 02_tfidf_sklearn.ipynb ────────────
def clean_headline(text: str) -> str:
    if not isinstance(text, str): return ''
    text = text.strip()
    text = text.replace('\u2018',"'").replace('\u2019',"'")
    text = text.replace('\u201c','"').replace('\u201d','"')
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r"[^a-zA-Z0-9\s']", ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

df['headline_clean'] = df['headline'].apply(clean_headline)

# ── Same 80/20 stratified split — RANDOM_STATE=42 ────────────────────────────
X = df['headline_clean'].values
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'   Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# ── TF-IDF — identical config to TFIDF_CONFIGS['tfidf_bigram'] in 02 notebook ─
print('⏳ Fitting TF-IDF vectorizer (bigram, max_features=50000)...')
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    stop_words='english',
    min_df=2,
    max_df=0.95,
)
X_train_vec = tfidf.fit_transform(X_train)

# ── SVM — identical hyperparams to 02_tfidf_sklearn.ipynb ────────────────────
print('⏳ Training Linear SVM (C=1.0, calibrated)...')
t0 = time.time()
base_svm  = LinearSVC(C=1.0, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
svm_model = CalibratedClassifierCV(base_svm, cv=3)
svm_model.fit(X_train_vec, y_train)
print(f'   Trained in {time.time()-t0:.1f}s')

# Quick sanity check against known results
X_test_vec = tfidf.transform(X_test)
test_acc   = accuracy_score(y_test, svm_model.predict(X_test_vec))
print(f'   Test accuracy (should be ~91.8%): {test_acc:.4f}')
print('✅ TF-IDF + SVM ready.')

## Section 3 — Load BERT Checkpoint

In [ ]:
# ── Load the BERT checkpoint saved by 03_bert_finetune.ipynb ─────────────────
# The checkpoint is saved to outputs/method3/checkpoint/ by the training loop.
# If you are running this in a fresh session, mount Google Drive first:
#   from google.colab import drive; drive.mount('/content/drive')
# Then update M3_CKPT above to the path on your Drive.

# Variables used in 03_bert_finetune.ipynb
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 64

BERT_AVAILABLE = False

if M3_CKPT.exists():
    print(f'⏳ Loading BERT from {M3_CKPT}...')
    bert_tokenizer = BertTokenizerFast.from_pretrained(str(M3_CKPT))
    bert_model     = BertForSequenceClassification.from_pretrained(str(M3_CKPT))
    bert_model     = bert_model.to(DEVICE)
    bert_model.eval()
    BERT_AVAILABLE = True
    print(f'✅ BERT loaded ({sum(p.numel() for p in bert_model.parameters())/1e6:.0f}M params) → {DEVICE}')
else:
    print(f'⚠️  BERT checkpoint not found at {M3_CKPT}')
    print('   Demo will run Methods 1 & 2 only.')
    print('   To add BERT: run 03_bert_finetune.ipynb first, then re-run this cell.')

## Section 4 — Rule Engine (from 01_rule_based.ipynb)

In [ ]:
# ── Rule engine — exact copy from 01_rule_based.ipynb Section 3 ──────────────
# (Duplicated here so 04_demo.ipynb is self-contained)

nlp = spacy.load('en_core_web_sm', disable=['parser'])

HYPERBOLE_WORDS = {
    'literally','absolutely','totally','completely','utterly','definitely',
    'obviously','clearly','certainly','undeniably','unbelievably','insanely',
    'hilariously','miraculously','shockingly','astoundingly','breathtakingly',
    'spectacular','phenomenal','incredible','unreal','surreal','mind-blowing',
    'greatest','best','worst','perfect','flawless','genius','brilliant',
    'amazing','fantastic','enormous','massive','gigantic','monumental',
    'unprecedented','revolutionary','earth-shattering','game-changing',
    'perfectly','beautifully','wonderfully','gloriously',
}
POSITIVE_WORDS = {
    'good','great','wonderful','excellent','superb','outstanding','brilliant',
    'fantastic','amazing','incredible','perfect','best','beautiful','love',
    'enjoy','happy','glad','pleased','delighted','celebrate','praise',
    'honor','award','heroic','noble','brave','generous','kind','helpful',
    'inspiring','admire','commend','applaud','heartwarming','joyful',
    'thrilled','excited','elated','overjoyed','blessed',
}
NEGATIVE_ENTITIES = {
    'trump','isis','taliban','congress','senate','politician','politicians',
    'government','lobbyist','ceo','corporate','corporation','bank','banks',
    'insurance','pharmaceutical','lawyer','lawyers','bureaucrat','bureaucrats',
    'republican','democrat','republicans','democrats',
}
ONION_PATTERNS = [
    r'\barea\s+(man|woman|teen|child|boy|girl|resident|father|mother)\b',
    r'\blocal\s+(man|woman|teen|child|boy|girl|resident|father|mother)\b',
    r'\bnation.?s?\s+(man|woman|children|citizens|workers|taxpayers|seniors)\b',
    r'\breport(s|edly)?:',
    r'\bstudies?\s+show\b',
    r'\bnew\s+report\b',
    r'\bexperts?\s+(say|claim|warn|reveal|suggest|confirm|agree)\b',
    r'\bscientists?\s+(say|claim|warn|reveal|suggest|discover|confirm)\b',
    r'\breport\s+finds?\b',
    r'\bnation.?s\b',
]
INCONGRUITY_PAIRS = [
    ({'excited','thrilled','happy','glad','pleased','delighted','overjoyed'},
     {'cancer','death','dying','bankrupt','fired','killed','arrested',
      'disaster','war','bomb','shooting','crash','poverty','recession',
      'unemployed','evicted','homeless','disease','pandemic'}),
    ({'celebrate','celebrates','honor','honors','awards','praises'},
     {'failure','loss','defeat','worst','mistake','error','blunder','flop'}),
]

def score_headline(text: str) -> dict:
    doc = nlp(text)
    tl  = {t.text.lower() for t in doc}
    ll  = {t.lemma_.lower() for t in doc}
    tlo = text.lower()
    signals = {}
    signals['hyperbole']            = 1 if len(tl & HYPERBOLE_WORDS) >= 2 else 0
    signals['ironic_quotes']        = 1 if any(c in text for c in ['"',"'",'\u2018','\u2019','\u201c','\u201d']) else 0
    signals['exclamation']          = 1 if '!' in text else 0
    signals['onion_pattern']        = 1 if any(re.search(p, tlo) for p in ONION_PATTERNS) else 0
    has_pos    = bool(ll & POSITIVE_WORDS)
    ner_tokens = {e.text.lower() for e in doc.ents} | tl
    signals['sentiment_incongruity'] = 1 if (has_pos and bool(ner_tokens & NEGATIVE_ENTITIES)) else 0
    neg_idx = {i for i,t in enumerate(doc) if t.lower_ in {'not',"n't",'never','no'}}
    pos_idx = {i for i,t in enumerate(doc) if t.lemma_.lower() in POSITIVE_WORDS}
    signals['negation_positive']    = 1 if any(abs(n-p)<=2 for n in neg_idx for p in pos_idx) else 0
    explicit = 0
    for ps, ns in INCONGRUITY_PAIRS:
        if (ll & ps) and (ll & ns): explicit = 1; break
    signals['explicit_incongruity'] = explicit
    total_score = sum(signals.values())
    return {'signals': signals, 'score': total_score, 'prediction': 1 if total_score >= 2 else 0}

print('✅ Rule engine loaded.')

## Section 5 — Predict Functions

In [ ]:
# ── Method 1: Rule-Based ──────────────────────────────────────────────────────
def predict_rule(text: str):
    """Returns (prediction, confidence_0_to_1, fired_signals_list)."""
    res     = score_headline(text)
    pred    = res['prediction']
    # Normalise score to [0,1] using max possible signals (7)
    conf    = min(res['score'] / 7.0, 1.0)
    fired   = [k for k, v in res['signals'].items() if v == 1]
    return pred, conf, fired


# ── Method 2: TF-IDF + SVM ────────────────────────────────────────────────────
def predict_tfidf(text: str):
    """Returns (prediction, confidence, p_sarcastic)."""
    cleaned = clean_headline(text)
    vec     = tfidf.transform([cleaned])
    prob    = svm_model.predict_proba(vec)[0]
    pred    = int(np.argmax(prob))
    return pred, float(prob[pred]), float(prob[1])


# ── Method 3: BERT ────────────────────────────────────────────────────────────
def predict_bert(text: str):
    """Returns (prediction, confidence, p_sarcastic) or (None, None, None) if unavailable."""
    if not BERT_AVAILABLE:
        return None, None, None
    # Use clean_for_bert logic from 03_bert_finetune.ipynb (minimal cleaning)
    text_clean = re.sub(r'http\S+|www\.\S+', '', text).strip()
    text_clean = re.sub(r'\s+', ' ', text_clean)
    enc  = bert_tokenizer(
        text_clean, return_tensors='pt',
        truncation=True, max_length=MAX_LEN, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = bert_model(**enc).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    pred  = int(np.argmax(probs))
    return pred, float(probs[pred]), float(probs[1])


print('✅ Predict functions defined.')
# Quick smoke test
test_hl = "Area Man Absolutely Brilliant At Defending What He Imagines Constitution Says"
p1, c1, f1 = predict_rule(test_hl)
p2, c2, p2s = predict_tfidf(test_hl)
p3, c3, p3s = predict_bert(test_hl)
print(f'  M1 Rule : pred={p1}  conf={c1:.2f}  signals={f1}')
print(f'  M2 SVM  : pred={p2}  p(sarc)={p2s:.3f}')
print(f'  M3 BERT : pred={p3}  p(sarc)={p3s}' if p3 is not None else '  M3 BERT : not loaded')

## Section 6 — Interactive ipywidgets Demo

In [ ]:
# ── Example headlines — mix of sarcastic (S) and real (R) ────────────────────
EXAMPLES = [
    # Sarcastic (The Onion)
    "Area Man Passionate Defender Of What He Imagines Constitution To Be",
    "Trump absolutely brilliant at solving nation's healthcare crisis with one tweet",
    "Nation's Obesity Rates Skyrocket As McDonald's Introduces New 4000-Calorie Salad",
    "Local Man Heroically Manages To Not Say Thing He Was Thinking",
    "Report: Nation's Exhausted Parents Outsourcing Child-Rearing To Tablet Computers",
    "Man Clearly Overqualified For Position Of Worst Person You Have Ever Met",
    # Real (HuffPost)
    "Scientists discover new species of deep-sea fish in the Pacific Ocean",
    "Federal Reserve raises interest rates for third consecutive time this year",
    "New study shows regular exercise reduces risk of heart disease",
    "UN peacekeeping mission successfully restores order in conflict zone",
    "Cancer treatment funding increased by government for third year",
]

# ── Helper: probability bar HTML ──────────────────────────────────────────────
def prob_bar_html(prob, color='#2196F3', width=180):
    pct = int(prob * 100)
    return (
        f'<div style="display:inline-flex;align-items:center;gap:6px">'
        f'<div style="background:#e0e0e0;border-radius:3px;height:12px;width:{width}px">'
        f'<div style="background:{color};width:{pct}%;height:100%;border-radius:3px"></div></div>'
        f'<span style="font-size:12px;color:#555">{pct}%</span></div>'
    )

def verdict_badge_html(pred, conf):
    if pred == 1:
        bg, fg, txt = '#ffcdd2', '#b71c1c', '🎭 SARCASTIC'
    else:
        bg, fg, txt = '#c8e6c9', '#1b5e20', '📰 NOT Sarcastic'
    return (f'<span style="background:{bg};color:{fg};padding:3px 10px;'
            f'border-radius:12px;font-weight:bold;font-size:13px">{txt}</span>')

def build_result_html(headline, rb_pred, rb_conf, rb_signals,
                      sv_pred, sv_conf, sv_sprob,
                      bt_pred, bt_conf, bt_sprob):
    rb_bar = prob_bar_html(rb_conf,   '#FF5722')
    sv_bar = prob_bar_html(sv_sprob,  '#2196F3')
    rb_sig = ', '.join(rb_signals) if rb_signals else '<em style="color:#999">none</em>'

    bt_row = ''
    if bt_pred is not None:
        bt_bar = prob_bar_html(bt_sprob, '#9C27B0')
        bt_row = f'''
          <tr style="background:#fafafa">
            <td style="padding:9px"><b>Method 3: BERT</b><br><small style="color:#888">bert-base-uncased fine-tuned</small></td>
            <td style="padding:9px">{verdict_badge_html(bt_pred, bt_conf)}</td>
            <td style="padding:9px">{bt_bar}</td>
            <td style="padding:9px"><small>P(sarc)={bt_sprob:.3f}</small></td>
          </tr>'''
    else:
        bt_row = '''<tr><td colspan="4" style="padding:9px;color:#bbb;font-style:italic">
          Method 3: BERT — checkpoint not loaded (run 03_bert_finetune.ipynb first)</td></tr>'''

    return f'''
    <div style="font-family:sans-serif;border:1px solid #ddd;border-radius:8px;
                padding:16px;max-width:800px;box-shadow:0 2px 6px rgba(0,0,0,.07)">
      <h3 style="margin-top:0;color:#212121">Headline Analyzed</h3>
      <p style="background:#f5f5f5;padding:10px;border-radius:4px;
               font-style:italic;color:#333;margin:0 0 12px 0">"{headline}"</p>
      <table style="width:100%;border-collapse:collapse">
        <thead>
          <tr style="background:#1a237e;color:white">
            <th style="padding:9px;text-align:left;width:30%">Method</th>
            <th style="padding:9px;text-align:left;width:22%">Verdict</th>
            <th style="padding:9px;text-align:left;width:22%">P(sarcasm)</th>
            <th style="padding:9px;text-align:left">Details</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td style="padding:9px"><b>Method 1: Rule-Based</b><br><small style="color:#888">spaCy · 7 signals · threshold=2</small></td>
            <td style="padding:9px">{verdict_badge_html(rb_pred, rb_conf)}</td>
            <td style="padding:9px">{rb_bar}</td>
            <td style="padding:9px"><small style="color:#555">Signals: {rb_sig}</small></td>
          </tr>
          <tr style="background:#fafafa">
            <td style="padding:9px"><b>Method 2: TF-IDF + SVM</b><br><small style="color:#888">bigram · C=1.0 · calibrated</small></td>
            <td style="padding:9px">{verdict_badge_html(sv_pred, sv_conf)}</td>
            <td style="padding:9px">{sv_bar}</td>
            <td style="padding:9px"><small>P(sarc)={sv_sprob:.3f}</small></td>
          </tr>
          {bt_row}
        </tbody>
      </table>
    </div>'''

print('✅ Widget helpers defined.')

In [ ]:
# ── Build and display the interactive widget ──────────────────────────────────

style = {'description_width': 'initial'}

title_html = widgets.HTML(
    value=(
        '<div style="font-family:sans-serif">'
        '<h2 style="color:#1a237e;margin-bottom:4px">🔍 Sarcasm Detector — CSE3246 NLP Demo</h2>'
        '<p style="color:#666;font-size:14px;margin-top:0">'
        'Type any news headline → all 3 methods predict simultaneously.</p></div>'
    )
)

example_dd = widgets.Dropdown(
    options=['── Choose an example ──'] + EXAMPLES,
    description='Examples:',
    layout=widgets.Layout(width='92%'),
    style=style
)

text_input = widgets.Textarea(
    value=EXAMPLES[0],
    placeholder='Type any news headline here...',
    description='Headline:',
    layout=widgets.Layout(width='92%', height='68px'),
    style=style
)

analyze_btn = widgets.Button(
    description='  Analyze',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='140px', height='36px')
)
clear_btn = widgets.Button(
    description='  Clear',
    button_style='warning',
    icon='times',
    layout=widgets.Layout(width='120px', height='36px')
)

output_area = widgets.Output()


def on_dropdown_change(change):
    if change['new'] != '── Choose an example ──':
        text_input.value = change['new']


def on_analyze(b):
    headline = text_input.value.strip()
    if not headline:
        return
    with output_area:
        clear_output(wait=True)
        display(HTML('<p style="color:#888;font-style:italic">⏳ Running all methods...</p>'))

    rb_pred, rb_conf, rb_signals = predict_rule(headline)
    sv_pred, sv_conf, sv_sprob   = predict_tfidf(headline)
    bt_pred, bt_conf, bt_sprob   = predict_bert(headline)

    html = build_result_html(
        headline,
        rb_pred, rb_conf, rb_signals,
        sv_pred, sv_conf, sv_sprob,
        bt_pred, bt_conf, bt_sprob
    )
    with output_area:
        clear_output(wait=True)
        display(HTML(html))


def on_clear(b):
    text_input.value = ''
    with output_area:
        clear_output()


example_dd.observe(on_dropdown_change, names='value')
analyze_btn.on_click(on_analyze)
clear_btn.on_click(on_clear)

ui = widgets.VBox([
    title_html,
    example_dd,
    text_input,
    widgets.HBox([analyze_btn, clear_btn]),
    output_area
])

display(ui)

## Section 7 — Batch Evaluation on Curated Headlines

In [ ]:
# ── Curated batch — deliberately chosen hard cases + easy cases ───────────────
# Format: (headline, true_label, difficulty)

BATCH = [
    # Easy sarcastic (all methods should catch)
    ("Area Man Passionate Defender Of What He Imagines Constitution To Be",       1, 'easy-sarc'),
    ("Nation's Obesity Rates Skyrocket As McDonald's Introduces 4000-Calorie Salad", 1, 'easy-sarc'),
    # Easy real (all methods should reject)
    ("Scientists discover new species of fish in the Pacific Ocean",               0, 'easy-real'),
    ("Federal Reserve raises interest rates for third time this year",             0, 'easy-real'),
    # Hard sarcastic (TF-IDF / Rule may miss; BERT should catch)
    ("Local Man Heroically Manages To Not Say Thing He Was Thinking",             1, 'hard-sarc'),
    ("Man Clearly Overqualified For Position Of Worst Person You Have Ever Met",  1, 'hard-sarc'),
    ("Nation Finally Agrees On Something",                                         1, 'hard-sarc'),
    # Hard real (may trigger rule engine FP due to hyperbole)
    ("Scientists say new treatment absolutely transforms cancer outcomes",          0, 'hard-real'),
    ("Report finds record number of Americans living in poverty",                  0, 'hard-real'),
    ("Experts warn unprecedented climate change impacts accelerating globally",    0, 'hard-real'),
]

print('Running batch evaluation...')
rows = []
for headline, true_label, difficulty in BATCH:
    rb_p, _, _    = predict_rule(headline)
    sv_p, _, sv_s = predict_tfidf(headline)
    bt_p, _, _    = predict_bert(headline)
    rows.append({
        'Headline':    headline[:58] + '...' if len(headline) > 58 else headline,
        'True':        true_label,
        'Difficulty':  difficulty,
        'M1_Rule':     rb_p,
        'M2_SVM':      sv_p,
        'M3_BERT':     bt_p if bt_p is not None else 'N/A',
        'M1✓':         '✅' if rb_p == true_label else '❌',
        'M2✓':         '✅' if sv_p == true_label else '❌',
        'M3✓':         ('✅' if bt_p == true_label else '❌') if bt_p is not None else '—',
    })

batch_df = pd.DataFrame(rows)

m1_acc = (batch_df['M1_Rule'] == batch_df['True']).mean()
m2_acc = (batch_df['M2_SVM']  == batch_df['True']).mean()

print(batch_df[['Headline','True','Difficulty','M1✓','M2✓','M3✓']].to_string(index=False))
print(f'\nBatch accuracy  — M1 Rule: {m1_acc:.1%}  |  M2 SVM: {m2_acc:.1%}')
print('\n💡 Hard cases expose the gap between methods — exactly what the report should discuss.')

In [ ]:
# ── Load and print cross-method results from saved JSONs ──────────────────────
print('='*65)
print('  CROSS-METHOD SUMMARY (from saved result JSONs)')
print('='*65)

for path, label in [(M1_JSON, 'Method 1 — Rule-Based'), (M2_JSON, 'Method 2 — TF-IDF+SVM')]:
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        # M1 has metrics at top level; M2 has them nested under models.linear_svm.metrics
        if 'metrics' in data:
            m = data['metrics']
        else:
            m = data['models']['linear_svm']['metrics']
        print(f'  {label}')
        print(f'    Accuracy={m["Accuracy"]:.4f}  F1={m["F1"]:.4f}  MCC={m["MCC"]:.4f}  AUC={m.get("ROC_AUC","N/A")}')
    else:
        print(f'  {label} — JSON not found ({path})')

print('\n✅ Demo notebook complete!')
print('   → Fill in BERT results above once 03_bert_finetune.ipynb is run.')